# 5 · Context

A contextual dataset records the situation of each interaction next to the user
and the item: when it happened, on what device, in which mood. This guide derives
such fields from MovieLens-100K and follows them through WarpRec:

- declaring contextual columns, and the three kinds of field WarpRec reads them
  as: categorical, numeric and multi-valued;
- how each field is encoded, and where to read the encoding back;
- `Transactions` (one row per record) next to `Interactions` (the collapsed
  matrix), and the `duplicates` policy that decides how the matrix collapses;
- `training.sequence_pooling` for multi-valued fields;
- the contextual evaluation, one ranking per (user, item, context) row, and the
  `evaluation.mask_seen` policies;
- training a factorization machine on all four fields with the design pipeline,
  evaluating it in full and sampled mode, and comparing the pooling policies.

**Requirements:** `pip install warprec jupyter`. Runs in under two minutes on a
laptop CPU, most of it training the model five times.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

RUNS = Path("../runs/05-context")
RUNS.mkdir(parents=True, exist_ok=True)
folder = movielens_100k()  # downloads and converts it as guide 1 shows

## Contexts from MovieLens-100K

MovieLens has no contextual columns of its own, so four are derived, each known
at the moment a recommendation would be made:

- `daypart`: night, morning, afternoon or evening, from the timestamp read in US
  Central time (the users are spread over US time zones, so this is approximate);
- `weekend`: weekday or weekend;
- `tenure`: how long the user has been rating, `log1p` of the days since their
  first rating;
- `previous_genres`: the genres of the movie the user rated just before this one,
  `|`-separated, empty for a user's first rating.

The genres of the movie being rated would be the obvious multi-valued field, and
it would be wrong: a context has to be known before the item is chosen. WarpRec
ranks every test row against the whole catalogue under that row's context, and
trains on negatives that reuse the positive's context, so a field computed from
the target item tells the model which item is the answer. Attributes of the item
belong in side information (guide 4).

In [2]:
ratings = pd.read_csv(folder / "ratings.tsv", sep="\t")
items = pd.read_csv(folder / "items.tsv", sep="\t")

local = pd.to_datetime(ratings.timestamp, unit="s", utc=True).dt.tz_convert(
    "America/Chicago"
)
ratings["daypart"] = pd.cut(
    local.dt.hour,
    bins=[0, 6, 12, 18, 24],
    right=False,
    labels=["night", "morning", "afternoon", "evening"],
).astype(str)
ratings["weekend"] = np.where(local.dt.dayofweek >= 5, "weekend", "weekday")

first_rating = ratings.groupby("user_id").timestamp.transform("min")
ratings["tenure"] = np.log1p((ratings.timestamp - first_rating) / 86400).round(4)

ratings = ratings.sort_values(["user_id", "timestamp", "item_id"], ignore_index=True)
genres = ratings.item_id.map(items.set_index("item_id").genres)
ratings["previous_genres"] = genres.groupby(ratings.user_id).shift(1).fillna("")

ratings.to_csv(RUNS / "ratings-context.tsv", sep="\t", index=False)
ratings.iloc[:4]

,user_id,item_id,rating,timestamp,daypart,weekend,tenure,previous_genres
0,1,168,5,874965478,afternoon,weekday,0.0000,
1,1,172,5,874965478,afternoon,weekday,0.0000,Comedy
2,1,165,5,874965518,afternoon,weekday,0.0005,Action|Adventure|Drama|Romance|Sci-Fi|War
3,1,156,4,874965556,afternoon,weekday,0.0009,Drama


## Declaring contextual columns

Contexts live in the interaction file itself, one value per row. Through the
API they are named with `context_labels` when the `Dataset` is built, and a
column that holds several values per cell is given its separator in
`context_separators`. The split below holds out each user's last rating
(guide 3 covers splitting).

In [3]:
from warprec.data import Dataset
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter

logger.disable("warprec")
train, _, test = Splitter().split_transaction(
    ratings, test=SplitSpec(strategy="temporal_leave_k_out", k=1)
)

logger.enable("warprec")
dataset = Dataset(
    train_data=train,
    eval_data=test,
    rating_type="explicit",
    rating_label="rating",
    timestamp_label="timestamp",
    context_labels=["daypart", "weekend", "tenure", "previous_genres"],
    context_separators={"previous_genres": "|"},
)
logger.disable("warprec")

08-10-2026 12:22:50 - WarpRec - 📝 Context 'daypart': found 4 unique values.


08-10-2026 12:22:51 - WarpRec - 📝 Context 'weekend': found 2 unique values.


08-10-2026 12:22:51 - WarpRec - 📝 Context 'tenure': numeric, kept as a value.


08-10-2026 12:22:51 - WarpRec - 📝 Context 'previous_genres': multi-valued, 19 distinct values, up to 6 per row.


08-10-2026 12:22:51 - WarpRec - 📝 -------------------------------Train set--------------------------------


08-10-2026 12:22:51 - WarpRec - 📊 Number of users: 943      Number of items: 1680      Transactions: 99057


08-10-2026 12:22:51 - WarpRec - 📝 ------------------------------------------------------------------------


08-10-2026 12:22:51 - WarpRec - 📝 -------------------------------Test set------------------------------


08-10-2026 12:22:51 - WarpRec - 📊 Number of users: 941      Number of items: 473      Transactions: 941


08-10-2026 12:22:51 - WarpRec - 📝 ---------------------------------------------------------------------


The log says how each column was read. A column is one of three kinds of field:

- **categorical**, the default: every distinct value gets an index;
- **numeric**, when the column reads as a float: the value is kept and scales
  one embedding, so its ordering survives;
- **multi-valued**, when it has a separator: the values share one vocabulary
  and are pooled into one vector.

The kind follows the column's type. An integer column, such as a year or a
count of days, is categorical and gets one token per distinct value; declare it
`float32` to have it treated as a quantity.

In a configuration the same declaration sits in the `reader`:
`labels.context_labels` names the columns, `dtypes.context_types` gives their
types and `dtypes.context_separators` the separators.

In [4]:
print(Path("configs/context.yml").read_text())

# Every contextual field the guide derives, declared in the reader.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../runs/05-context/ratings-context.tsv
  rating_type: explicit
  # How repeated (user, item) rows collapse into the interaction matrix.
  duplicates: max
  labels:
    context_labels: [daypart, weekend, tenure, previous_genres]
  dtypes:
    context_types:
      daypart: str
      weekend: str
      tenure: float32 # a measurement, not a category
      previous_genres: str
    context_separators:
      previous_genres: "|" # several values in one cell
splitter:
  test_splitting:
    strategy: temporal_leave_k_out
    k: 1
training:
  # How the values of a multi-valued field are combined: mean, sum or max.
  sequence_pooling: mean
# This file is read for its data sections only; guide 9 covers models.
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [5]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

config = load_design_configuration("configs/context.yml")
from_yaml, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
from_yaml.get_context_dims() == dataset.get_context_dims()

True

## How each field is encoded

The vocabulary of every categorical and multi-valued field is built **from the
training split only**, in sorted order, starting at 1. Index 0 is reserved: it
stands for a value never seen in training and, in a multi-valued field, for
padding. `get_context_maps` returns the vocabulary; a numeric field has none.

In [6]:
maps = dataset.get_context_maps()
{field: dict(list(values.items())[:5]) for field, values in maps.items()}

{'daypart': {'afternoon': 1, 'evening': 2, 'morning': 3, 'night': 4},
 'weekend': {'weekday': 1, 'weekend': 2},
 'previous_genres': {'Action': 1,
  'Adventure': 2,
  'Animation': 3,
  "Children's": 4,
  'Comedy': 5}}

`get_context_dims` gives the size of each field's embedding table: the
vocabulary plus index 0, and 1 for a numeric field, which owns a single
embedding. The models read these from `info()`, which a contextual dataset
extends with four keys:

In [7]:
{
    key: value
    for key, value in dataset.info().items()
    if key.startswith(("context", "seq"))
}

{'context_dims': {'daypart': 5,
  'weekend': 3,
  'tenure': 1,
  'previous_genres': 20},
 'context_types': {'daypart': 'token',
  'weekend': 'token',
  'tenure': 'float',
  'previous_genres': 'seq'},
 'context_max_len': {'previous_genres': 6},
 'sequence_pooling': 'mean'}

`context_types` is how a model tells the kinds apart (`token`, `float`, `seq`)
and `context_max_len` is the most values any row of a multi-valued field holds,
here a movie with six genres.

The encoded rows are held in the dataset's `Transactions`. With a multi-valued
field among them the context array gains a third dimension, one slot per value,
padded with 0; a categorical or numeric field uses only the first slot. The
first training row whose previous movie had three genres, field by field:

In [8]:
users, items_idx, _, contexts = dataset.train_transactions.get_arrays()
row = int(np.argmax((contexts[:, 3] > 0).sum(axis=1) == 3))
print(contexts.shape)
pd.DataFrame(contexts[row], index=dataset.train_transactions.context_labels)

(99057, 4, 6)


,0,1,2,3,4,5
daypart,4.0000,0.0,0.0,0.0,0.0,0.0
weekend,1.0000,0.0,0.0,0.0,0.0,0.0
tenure,3.7474,0.0,0.0,0.0,0.0,0.0
previous_genres,1.0000,8.0,16.0,0.0,0.0,0.0


A value that appears only in the evaluation split is mapped to 0 rather than
given a new index, since no model was trained on it. The small frame below has
one: nobody watched anything in the afternoon during training. WarpRec reports
it for a categorical field; an unknown value inside a multi-valued cell (here
`Horror`) becomes 0 as well.

In [9]:
toy_train = pd.DataFrame(
    {
        "user_id": [1, 1, 1, 2, 2],
        "item_id": [10, 10, 11, 10, 12],
        "rating": [3.0, 5.0, 4.0, 2.0, 4.0],
        "daypart": ["morning", "evening", "evening", "night", "morning"],
        "genres": ["Drama", "Comedy|Drama", "", "Action", "Action|Comedy"],
    }
)
toy_test = pd.DataFrame(
    {
        "user_id": [1, 2],
        "item_id": [12, 11],
        "rating": [4.0, 3.0],
        "daypart": ["afternoon", "evening"],
        "genres": ["Horror|Drama", "Comedy"],
    }
)

logger.enable("warprec")
toy = Dataset(
    toy_train,
    toy_test,
    rating_type="explicit",
    rating_label="rating",
    context_labels=["daypart", "genres"],
    context_separators={"genres": "|"},
)
logger.disable("warprec")
print(toy.get_context_maps())
toy.eval_transactions.get_arrays()[3]

08-10-2026 12:22:51 - WarpRec - 📝 Context 'daypart': found 3 unique values.


08-10-2026 12:22:51 - WarpRec - 📝 Context 'genres': multi-valued, 3 distinct values, up to 2 per row.


08-10-2026 12:22:51 - WarpRec - 📢 Context 'daypart': found 1 unknown values in Eval set (mapped to 0).


08-10-2026 12:22:51 - WarpRec - 📝 ---------------------------Train set---------------------------


08-10-2026 12:22:51 - WarpRec - 📊 Number of users: 2      Number of items: 3      Transactions: 5


...

08-10-2026 12:22:51 - WarpRec - 📝 ----------------------------Test set---------------------------


08-10-2026 12:22:51 - WarpRec - 📊 Number of users: 2      Number of items: 2      Transactions: 2


08-10-2026 12:22:51 - WarpRec - 📝 ---------------------------------------------------------------


{'daypart': {'evening': 1, 'morning': 2, 'night': 3}, 'genres': {'Action': 1, 'Comedy': 2, 'Drama': 3}}


array([[[0., 0.],
        [0., 3.]],

       [[1., 0.],
        [2., 0.]]], dtype=float32)

## `Transactions` and `Interactions`

The toy frame also has a pair seen twice: user 1 watched item 10 in the morning
and again in the evening, with different ratings. A contextual dataset keeps
both views of the training split:

- `train_transactions`, a `Transactions`, holds one row per record, so the pair
  appears twice with its two contexts. Context-aware models train on these rows.
- `train_set`, an `Interactions`, is the user × item matrix every other model
  reads. It has one cell per pair, so the two records collapse into one.

In [10]:
users, items_idx, values, contexts = toy.train_transactions.get_arrays()
user_map, item_map = toy.get_mappings()
pair = (users == user_map[1]) & (items_idx == item_map[10])
inverse_daypart = {i: v for v, i in toy.get_context_maps()["daypart"].items()}
print(len(toy.train_transactions), "rows,", toy.train_set.get_sparse().nnz, "cells")
pd.DataFrame(
    {
        "rating": values[pair],
        "daypart": [inverse_daypart[int(c)] for c in contexts[pair][:, 0, 0]],
    }
)

5 rows, 4 cells


,rating,daypart
0,3.0,morning
1,5.0,evening


How a pair collapses into its cell is the `duplicates` policy, `reader.duplicates`
in a configuration: `max` (the default), `mean`, `first`, `last` or `sum`. It
changes only the matrix; the rows are kept as they are. With implicit feedback
every policy but `sum` gives a 1, and `sum` counts the records.

In [11]:
pd.DataFrame(
    {
        policy: [
            Dataset(
                toy_train,
                rating_type="explicit",
                rating_label="rating",
                duplicates=policy,
            ).train_set.get_sparse()[user_map[1], item_map[10]]
        ]
        for policy in ["max", "mean", "first", "last", "sum"]
    },
    index=["cell (user 1, item 10)"],
)

,max,mean,first,last,sum
"cell (user 1, item 10)",5.0,4.0,3.0,5.0,8.0


MovieLens-100K has no repeated pair, so on it the two views hold the same
number of records:

In [12]:
len(dataset.train_transactions), dataset.train_set.get_sparse().nnz

(99057, 99057)

## Pooling a multi-valued field

Every field contributes exactly one vector to a context-aware model, so the
values of a multi-valued field are combined into one. `training.sequence_pooling`
chooses how:

- `mean` (the default) averages the embeddings of the values, padding excluded,
  so a field's contribution does not grow with the number of values it holds;
  it is the embedding counterpart of the normalised multi-hot encoding
  factorization machines are usually defined over;
- `sum` adds them;
- `max` takes the element-wise maximum.

A field's first-order weight is pooled the same way as its embedding. The
policy travels to the model through `info()["sequence_pooling"]`, shown above;
through the API it is the `sequence_pooling` argument of `Dataset`, and in a
configuration it sits under `training`. The model section below trains the
same model with each of the three.

## The contextual evaluation

A top-N evaluation ranks once per user. A contextual one ranks once per **test
row**: the model scores the whole catalogue for that user *in that row's
context*, and the row's item is the one relevant answer. The model below uses
all four fields; this is its configuration.

In [13]:
print(Path("configs/fm.yml").read_text())

# A factorization machine over the user, the item and all four contextual fields.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../runs/05-context/ratings-context.tsv
  rating_type: implicit
  labels:
    context_labels: [daypart, weekend, tenure, previous_genres]
  dtypes:
    context_types:
      daypart: str
      weekend: str
      tenure: float32
      previous_genres: str
    context_separators:
      previous_genres: "|"
splitter:
  test_splitting:
    strategy: temporal_leave_k_out
    k: 1
training:
  sequence_pooling: mean
models:
  FM:
    embedding_size: 32
    reg_weight: 0.0001
    batch_size: 2048
    epochs: 3
    learning_rate: 0.001
    neg_samples: 1
    optimization:
      device: cpu
      num_workers: 0
evaluation:
  top_k: [10]
  metrics: [nDCG, HitRate]
  # MovieLens never repeats a (user, item) pair, so exclude every seen item.
  mask_seen: pair
general:
  device: cpu



The evaluation loader yields exactly that, (user, item, context) per test row,
with the context laid out as in `Transactions`:

In [14]:
config = load_design_configuration("configs/fm.yml")
main, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
loader = main.get_contextual_evaluation_dataloader()
user_batch, item_batch, context_batch = next(iter(loader))
print(len(loader.dataset), "rankings, one per test row")
print("context batch", tuple(context_batch.shape))
print("first row: user", int(user_batch[0]), "item", int(item_batch[0]))
pd.DataFrame(context_batch[0].numpy(), index=main.train_transactions.context_labels)

941 rankings, one per test row
context batch (941, 4, 6)
first row: user 0 item 73


,0,1,2,3,4,5
daypart,2.0000,0.0,0.0,0.0,0.0,0.0
weekend,1.0000,0.0,0.0,0.0,0.0,0.0
tenure,5.1483,0.0,0.0,0.0,0.0,0.0
previous_genres,5.0000,14.0,0.0,0.0,0.0,0.0


Per-user metrics are averaged over a user's rows, then over users. With one
held-out rating per user, as here, every user has exactly one row.

Before ranking, items the user has already seen are excluded. Which ones is
`evaluation.mask_seen`:

| policy | excludes |
|---|---|
| `auto` (default) | `context` when the dataset has contextual columns, `pair` otherwise |
| `context` | the items the user saw **in exactly this context** |
| `pair` | every item the user has seen, in any context |
| `none` | nothing |

`context` is the default for contextual data because a movie watched on a
weekday morning can be a fair recommendation for a weekend evening.
`configs/fm.yml` sets `pair`; the model section shows why.

## Training a context-aware model

`design_pipeline` runs a configuration end to end: it reads, splits, encodes the
contexts, trains and logs the evaluation. Each training row is a positive and is
paired with `neg_samples` sampled items in the same context. Lightning's
warnings are silenced, and its model summary and progress bar are captured
away so the output is WarpRec's log.

In [15]:
import logging
import warnings

from IPython.utils.capture import capture_output

from warprec.pipelines import design_pipeline

logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")

logger.enable("warprec")
with capture_output():
    design_pipeline("configs/fm.yml")

08-10-2026 12:22:52 - WarpRec - 📝 Starting the Design Pipeline.


08-10-2026 12:22:52 - WarpRec - 📝 Reading design configuration file in: configs/fm.yml


08-10-2026 12:22:52 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:22:52 - WarpRec - 📝 Starting test splitting process with temporal_leave_k_out splitting strategy.


08-10-2026 12:22:52 - WarpRec - 📝 Test splitting completed in : 0.01s


...

08-10-2026 12:23:03 - WarpRec - 📝 | Cutoff   |      nDCG |   HitRate |


08-10-2026 12:23:03 - WarpRec - 📝 +==========+===========+===========+


08-10-2026 12:23:03 - WarpRec - 📝 | Top@10   | 0.0525416 |  0.105207 |


08-10-2026 12:23:03 - WarpRec - 📝 +----------+-----------+-----------+


08-10-2026 12:23:03 - WarpRec - ✅ Design pipeline executed successfully. WarpRec is shutting down.


The log lists the four fields as they were encoded and ends with the full
evaluation: every test row ranked against the whole catalogue.

Sampled evaluation ranks each test row against `num_negatives` items instead,
drawn among those the user never rated in training, in any context. The only
change is in `evaluation`:

In [16]:
print(Path("configs/fm-sampled.yml").read_text().split("evaluation:")[1])


  top_k: [10]
  metrics: [nDCG, HitRate]
  # MovieLens never repeats a (user, item) pair, so exclude every seen item.
  mask_seen: pair
  # Rank each test item against 99 items the user never rated.
  strategy: sampled
  num_negatives: 99
general:
  device: cpu



In [17]:
with capture_output():
    design_pipeline("configs/fm-sampled.yml")
logger.disable("warprec")

08-10-2026 12:23:03 - WarpRec - 📝 Starting the Design Pipeline.


08-10-2026 12:23:03 - WarpRec - 📝 Reading design configuration file in: configs/fm-sampled.yml


08-10-2026 12:23:03 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:23:04 - WarpRec - 📝 Starting test splitting process with temporal_leave_k_out splitting strategy.


08-10-2026 12:23:04 - WarpRec - 📝 Test splitting completed in : 0.01s


...

08-10-2026 12:23:17 - WarpRec - 📝 | Cutoff   |     nDCG |   HitRate |


08-10-2026 12:23:17 - WarpRec - 📝 +==========+==========+===========+


08-10-2026 12:23:17 - WarpRec - 📝 | Top@10   | 0.266978 |  0.456961 |


08-10-2026 12:23:17 - WarpRec - 📝 +----------+----------+-----------+


08-10-2026 12:23:17 - WarpRec - ✅ Design pipeline executed successfully. WarpRec is shutting down.


Trained with the same seed and settings, the model reaches nDCG@10 0.2670 and
HitRate@10 0.4570 against 100 candidates, where the full ranking gave 0.0525
and 0.1052 against all 1680 items. `mask_seen` plays no part here: the
negatives are already items the user never rated.

### `sequence_pooling` end to end

Through the API the same configuration can be taken apart, which keeps the
trained model in hand. The model is built from the `models` section and the
dataset's `info()`, and receives the training `Transactions` it draws its rows
from. `training.sequence_pooling` reaches the model as
`info()["sequence_pooling"]`, so overriding that key trains the same model
under each policy. All three are evaluated by the evaluator `fm.yml` describes:

In [18]:
import lightning as L

from warprec.evaluation import build_evaluator
from warprec.utils.registry import model_registry


def train_fm(pooling: str):
    model = model_registry.get(
        "FM",
        params=config.models["FM"],
        info={**main.info(), "sequence_pooling": pooling},
        interactions=main.train_set,
        transactions=main.train_transactions,
        seed=42,
    )
    trainer = L.Trainer(
        max_epochs=model.epochs,
        accelerator="cpu",
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    trainer.fit(model, model.get_dataloader(main.train_set, main.train_session))
    return model


def evaluate(model, evaluator) -> dict:
    evaluator.evaluate(model, loader, "full", main)
    results = evaluator.compute_results()[10]
    return {name: float(v.nanmean()) for name, v in results.items()}


models = {pooling: train_fm(pooling) for pooling in ["mean", "sum", "max"]}
evaluator = build_evaluator(config.evaluation, main)
pd.DataFrame({p: evaluate(m, evaluator) for p, m in models.items()}).T.round(4)

,nDCG,HitRate
mean,0.0525,0.1052
sum,0.0517,0.1020
max,0.0523,0.1052


The `mean` row reproduces the design pipeline's full evaluation above (nDCG@10
0.0525). After three epochs the three policies end close together: `sum` is
lowest on both metrics (0.0517, 0.1020), `max` matches `mean` on HitRate
(0.1052).

### `mask_seen` on the same model

The evaluator takes the masking policy as `mask_seen`. The `mean` model,
evaluated under each of the four:

In [19]:
from warprec.evaluation.evaluator import Evaluator

model = models["mean"]
rows = {}
for policy in ["auto", "context", "pair", "none"]:
    policy_evaluator = Evaluator(
        list(config.evaluation.metrics),
        list(config.evaluation.top_k),
        train_set=main.train_set.get_sparse(),
        mask_seen=policy,
    )
    rows[policy] = evaluate(model, policy_evaluator)
_, context_ids = main.train_transactions.get_context_index()
print(
    len(context_ids),
    "distinct contexts in",
    len(main.train_transactions),
    "training rows",
)
pd.DataFrame(rows).T.round(4)

72931 distinct contexts in 99057 training rows


,nDCG,HitRate
auto,0.0267,0.0595
context,0.0267,0.0595
pair,0.0525,0.1052
none,0.0267,0.0595


`auto` resolved to `context`, yet the `context` row equals `none`: 0.0267
against 0.0525 under `pair`. With the numeric `tenure` among the fields, 72931
of the 99057 training rows carry a context of their own, so "seen in this
exact context" almost never matches and nothing is masked. MovieLens never
repeats a pair either, so a held-out movie is never one the user has rated,
and every rated movie left in the ranking competes with the answer. On this
data `pair` matches how the test set was built, which is why `configs/fm.yml`
sets it; `context` suits coarse contexts and data where users do come back to
an item in a new situation. Report the policy with the numbers.

## Recommendation files

Writing recommendations is refused for a context-aware model. A recommendation
file answers "what should this user see", with no situation attached, and these
models only score a user in a situation. This is deliberate, final behaviour:
the writer raises `NotImplementedError`. The training pipeline's
`meta.save_recs: true` goes through the same writer, so it stops there too,
once the model has been trained and evaluated. Evaluate these models instead,
which supplies the contexts.

In [20]:
from warprec.data.writer import LocalWriter

writer = LocalWriter(dataset_name="ml-100k", local_path=str(RUNS / "experiments"))
try:
    writer.write_recs(model, main, k=10)
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: FM is context-aware and cannot write recommendations: a recommendation file carries no context to score against.


## Summary

| key | what it does |
|---|---|
| `reader.labels.context_labels` | the contextual columns of the interaction file |
| `reader.dtypes.context_types` | the type of each; a float type makes the field numeric |
| `reader.dtypes.context_separators` | the separator of a multi-valued field |
| `reader.duplicates` | how repeated (user, item) rows collapse into the matrix: `max`, `mean`, `first`, `last`, `sum` |
| `training.sequence_pooling` | how a multi-valued field becomes one vector: `mean`, `sum`, `max` |
| `evaluation.mask_seen` | which seen items are excluded from each ranking: `auto`, `context`, `pair`, `none` |
| `evaluation.strategy`, `num_negatives` | `full` ranks the whole catalogue per row, `sampled` the item against `num_negatives` unseen ones |

The [Reader](https://warprec.readthedocs.io/en/latest/data-management/reader/)
page describes how contextual columns are read, the
[Context-Aware Recommenders](https://warprec.readthedocs.io/en/latest/recommenders/context/)
page lists the eight models and their parameters, and the
[Evaluation configuration](https://warprec.readthedocs.io/en/latest/configuration/evaluation/)
page covers `mask_seen`. Guide 17 serves a context-aware model, where each
request carries its context.